In [1]:
import pandas as pd
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
import random
from itertools import product
import optuna
import numpy as np
from scipy.stats import norm

from stage1 import lasso_rolling_window, calculate_r_squared
from stage2 import estimate_kappa_curve_fit, compute_alm_returns, compute_stage2_r_squared
from grid_search import grid_search, estimate_single_config


c:\Users\jonat\anaconda3\envs\reddit_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load feature matrix and response variable
feature_matrix = pd.read_csv("../../data/merged_return_topic_data.csv", index_col=0, parse_dates=True)
response_variables = pd.read_csv("../../data/response.csv", index_col=0, parse_dates=True)

In [ ]:
def to_ar1_innovations(X: pd.DataFrame, min_obs: int = 30) -> pd.DataFrame:
    """Return AR(1) innovations (residuals) for each column of X."""
    X_innov = pd.DataFrame(index=X.index, columns=X.columns, dtype="float64")

    for col in X.columns:
        s = pd.to_numeric(X[col], errors="coerce")
        tmp = pd.DataFrame({"x": s, "x_lag1": s.shift(1)}).dropna()
        if len(tmp) < min_obs or tmp["x"].nunique() < 3 or tmp["x_lag1"].nunique() < 3:
            continue

        res = sm.OLS(tmp["x"], sm.add_constant(tmp["x_lag1"])).fit()
        X_innov.loc[tmp.index, col] = res.resid

    return X_innov

In [ ]:
# set seed
random.seed(42)

# select a response variable (either marekt return or sp500 return)
y = response_variables['vwretx']  # or 'sprtrn' for SP500 returns or vwretx

# # transform returns to log
y = np.log(y+1)

# create feature matrix
X = feature_matrix.copy()

# Separate topic (name) and stock (numeric) columns
topic_cols = [col for col in X.columns if not str(col).isdigit()]
stock_cols = [col for col in X.columns if str(col).isdigit()]

# User-selected number of each 
num_topics = len(topic_cols)  
num_stocks = 0

# Randomly sample (without replacement), limited by available count
selected_topics = random.sample(topic_cols, min(num_topics, len(topic_cols)))
selected_stocks = random.sample(stock_cols, min(num_stocks, len(stock_cols)))

# Final filtered dataframe
X = X[selected_topics + selected_stocks]

# Convert stock returns to log returns: log(1+r)
X[selected_stocks] = np.log(X[selected_stocks] + 1)

# ensure that all indices align
common_index = X.index.intersection(y.index)
X = X.loc[common_index]
y = y.loc[common_index]

In [7]:
from itertools import product
import pandas as pd
from joblib import Parallel, delayed
from tqdm import tqdm

def grid_search(
    X,
    y,
    param_grid,
    verbose=True,
    n_jobs=-1,
    backend="loky",
    prefer=None,
    return_details=True,
):
    """
    Parallel grid search.

    Parameters
    ----------
    return_details : bool, default True
        If False, skip collecting and concatenating window-level details
        (MUCH faster and lower memory usage).

    Returns
    -------
    results_df : pd.DataFrame
        Summary metrics for each configuration.
    coefficients_df : pd.DataFrame or None
        Detailed coefficients if return_details=True, else None.
    """

    combos = list(product(
        param_grid["window_sizes"],
        param_grid["n_lags"],
        param_grid["lambdas"],
    ))

    if verbose:
        print(f"Testing {len(combos)} configurations...")

    # -------------------------------------------------
    # Wrapper to prevent single failure from killing run
    # -------------------------------------------------
    def _safe_run(args):
        w, L, lam = args
        try:
            return estimate_single_config(X, y, w, L, lam)
        except Exception as e:
            if verbose:
                print(f"❌ Failed config (w={w}, L={L}, λ={lam}): {e}")
            return None

    iterator = tqdm(combos, desc="Grid search") if verbose else combos

    # -------------------------------------------------
    # Parallel execution
    # -------------------------------------------------
    results = Parallel(n_jobs=n_jobs, backend=backend, prefer=prefer)(
        delayed(_safe_run)(args) for args in iterator
    )

    # -------------------------------------------------
    # Aggregate results
    # -------------------------------------------------
    summary_list = []
    details_list = [] if return_details else None

    for res in results:
        if res is None:
            continue

        summary_list.append(res.get("summary", {}))

        if return_details:
            det = res.get("details", None)
            if det is not None and not det.empty:
                details_list.append(det)

    # -------------------------------------------------
    # Build summary DataFrame
    # -------------------------------------------------
    results_df = pd.DataFrame(summary_list)

    if not results_df.empty and "r2_oos_stage2" in results_df.columns:
        results_df = results_df.sort_values(
            "r2_oos_stage2", ascending=False
        ).reset_index(drop=True)

    # -------------------------------------------------
    # Build details DataFrame (optional)
    # -------------------------------------------------
    coefficients_df = None
    if return_details and details_list:
        coefficients_df = pd.concat(details_list, ignore_index=True)

    # -------------------------------------------------
    # Reporting
    # -------------------------------------------------
    if verbose:
        print("\n" + "=" * 80)
        print("GRID SEARCH COMPLETE")
        print("=" * 80)

        if "kappa" in results_df.columns:
            n_failed = results_df["kappa"].isna().sum()
            if n_failed > 0:
                print(f"⚠️  {n_failed}/{len(results_df)} configurations failed")

    return results_df, coefficients_df


## Grid Search

In [ ]:
# Objective Function
def objective_function(row):
    weight = 2 * (norm.cdf(abs(row['kappa_tstat'])) - 0.5)
    return row['r2_insample_stage2'] * weight
# -----------------------------
# Stopping criteria (NEW)
# -----------------------------
max_iterations = 20          # maximum refinement iterations
improvement_tol = 1e-6       # stop if best objective improves by less than this

# -------------------------------------------------
# Objective function: maximize R^2 only
# -------------------------------------------------
def objective_function(row, r2_col="r2_insample_stage2"):
    r2 = pd.to_numeric(row.get(r2_col, np.nan), errors="coerce")
    if not np.isfinite(r2):
        return -1.0
    return r2

if valid.empty:
    best_overall = None
    print("No valid configurations found.")
else:
    best_overall = valid.loc[valid["objective"].idxmax()]
    print(best_overall[["window_size", "n_lags", "lambda", "objective"]])

# -------------------------------------------------
# Iterative grid refinement
# -------------------------------------------------
results_all = []
prev_best = -np.inf

param_grid = {
    "window_sizes": [36, 52, 78, 104, 150, 200],
    "n_lags": [1, 4, 8, 12],
    "lambdas": [0.0001, 0.00001],
}

for iteration in range(5):

    # Run grid search
    summary_df, _ = grid_search(X, y, param_grid, verbose=True)

    # Compute objective
    summary_df["objective"] = summary_df.apply(objective_function, axis=1)
    results_all.append(summary_df)

    # Check for valid candidates
    valid_scores = summary_df.loc[summary_df["objective"] > -1.0, "objective"]
    if valid_scores.empty:
        break

    best_idx = valid_scores.idxmax()
    best_obj = valid_scores.max()

    # Convergence check
    if iteration > 0 and (best_obj - prev_best) < 1e-6:
        break

    prev_best = best_obj
    best = summary_df.loc[best_idx]

    # -------------------------------------------------
    # Refine grid around best point
    # -------------------------------------------------
    w = int(best["window_size"])
    l = int(best["n_lags"])
    lam = float(best["lambda"])

    param_grid = {
        "window_sizes": sorted(
            {int(w * f) for f in [0.75, 0.9, 1.0, 1.1, 1.25] if w * f >= 20}
        ),
        "n_lags": sorted({max(1, l - 2), l - 1, l, l + 1, l + 2}),
        "lambdas": lam * np.array([0.5, 0.75, 1.0, 1.25, 1.5]),
    }


# -------------------------------------------------
# Select best overall configuration
# -------------------------------------------------
final = pd.concat(results_all, ignore_index=True)
final["objective"] = final.apply(objective_function, axis=1)

best_overall = final.loc[final["objective"].idxmax()]
print(best_overall[["window_size", "n_lags", "lambda", "objective"]])


## Bayesian Optimization with optuna

For each hyperparameter triplet we do the following:

    - run stage1 and stage2
    - collect r2 in-sample stage2 and kappa
    - compute objective function: r2*kappa
    - let optuna try 150 random/learned samples to maximize the objective function

In [5]:
import numpy as np
import optuna
from optuna.samplers import TPESampler

TSTAT_MIN = 1.96
TSTAT_MAX = 60.0

# anti-degeneracy for stage 1
R2_STAGE1_MIN = 1e-5          # tune: start tiny
R2_STAGE1_WEIGHT = 0.05       # small tie-breaker, not a primary objective

# strong separation between feasible/infeasible
INFEASIBLE_BASE = -1000.0
FEASIBLE_BASE = 1.0

    # If t-stat is *absurdly* large, treat as pathological and prune.
    if HARD_PRUNE_TOO_LARGE and t > TSTAT_MAX:
        trial.set_user_attr("t_band_status", "pruned_too_large")
        raise optuna.TrialPruned()

    # If not significant, keep trial but strongly penalize (so TPE learns)
    if t < TSTAT_MIN:
        # distance below the threshold, scaled to be comparable across magnitudes
        dist = (TSTAT_MIN - t) / TSTAT_MIN  # in (0, +inf)
        penalty = SOFT_PENALTY_OUTSIDE * dist
        score = base / (1.0 + penalty)
        trial.set_user_attr("t_band_status", "below_min_soft_penalty")
        trial.set_user_attr("penalty", penalty)
        return float(score)

    # In band -> best region, no penalty
    trial.set_user_attr("t_band_status", "in_band")
    trial.set_user_attr("penalty", 0.0)
    return base


# -----------------------------
# Sampler + pruner
# -----------------------------
sampler = TPESampler(
    seed=42,
    n_startup_trials=30,
    multivariate=True,
    constant_liar=True
)

pruner = MedianPruner(n_startup_trials=20, n_warmup_steps=5)

study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)


def print_best_callback(study, trial):
    if trial.number % 10 == 0 and study.best_trial is not None:
        bt = study.best_trial
        print(
            f"Trial {trial.number}: best={study.best_value:.4f} "
            f"(t={bt.user_attrs.get('kappa_tstat', np.nan):.2f}, "
            f"R2={bt.user_attrs.get('r2_raw', np.nan):.4f}, "
            f"kappa={bt.user_attrs.get('kappa_raw', np.nan):.4f})"
        )


study.optimize(
    objective,
    n_trials=250,
    n_jobs=6,
    callbacks=[print_best_callback],
    show_progress_bar=True
)

print("\n" + "=" * 60)
print("OPTIMIZATION RESULTS")
print("=" * 60)
print(f"Best Score (objective): {study.best_value:.6f}")
print(f"Best Params: {study.best_params}")

best_trial = study.best_trial
print("\nBest Trial Metrics:")
print(f"  R²:      {best_trial.user_attrs.get('r2_raw', np.nan):.6f}")
print(f"  Kappa:   {best_trial.user_attrs.get('kappa_raw', np.nan):.6f}")
print(f"  t-stat:  {best_trial.user_attrs.get('kappa_tstat', np.nan):.6f}")
print(f"  Base:    {best_trial.user_attrs.get('base_score', np.nan):.6f}")
print(f"  Status:  {best_trial.user_attrs.get('t_band_status', '')}")

# -----------------------------
# Reporting: "best within band" explicitly
# -----------------------------
in_band = [
    t for t in study.trials
    if t.value is not None
    and t.state == optuna.trial.TrialState.COMPLETE
    and (t.user_attrs.get("kappa_tstat", -np.inf) >= TSTAT_MIN)
    and (t.user_attrs.get("kappa_tstat", np.inf) <= TSTAT_MAX)
]

in_band.sort(key=lambda tr: tr.user_attrs.get("base_score", -np.inf), reverse=True)

print("\n" + "=" * 60)
print(f"TOP 10 TRIALS IN ECONOMIC BAND ({TSTAT_MIN}–{TSTAT_MAX})")
print("=" * 60)
if in_band:
    for i, tr in enumerate(in_band[:10], 1):
        print(
            f"{i}. base={tr.user_attrs.get('base_score', np.nan):.6f} | "
            f"obj={tr.value:.6f} | "
            f"R²={tr.user_attrs.get('r2_raw', np.nan):.4f} | "
            f"kappa={tr.user_attrs.get('kappa_raw', np.nan):.4f} | "
            f"t={tr.user_attrs.get('kappa_tstat', np.nan):.2f} | "
            f"params={tr.params}"
        )
else:
    print("No completed trials ended up in the target t-stat band.")

# -----------------------------
# Diagnostics: pruned counts & where mass is
# -----------------------------
states = [tr.state for tr in study.trials]
n_pruned = sum(s == optuna.trial.TrialState.PRUNED for s in states)
n_complete = sum(s == optuna.trial.TrialState.COMPLETE for s in states)

below = sum(
    (tr.state == optuna.trial.TrialState.COMPLETE)
    and (tr.user_attrs.get("kappa_tstat", -np.inf) < TSTAT_MIN)
    for tr in study.trials
)
above_pruned = sum(
    (tr.state == optuna.trial.TrialState.PRUNED)
    and (tr.user_attrs.get("t_band_status", "") == "pruned_too_large")
    for tr in study.trials
)

print("\n" + "=" * 60)
print("DIAGNOSTICS")
print("=" * 60)
print(f"Complete trials: {n_complete}/{len(study.trials)}")
print(f"Pruned trials:   {n_pruned}/{len(study.trials)}")
print(f"Complete but t < {TSTAT_MIN}: {below}")
print(f"Pruned for t > {TSTAT_MAX}:  {above_pruned}")


c:\Users\jonat\anaconda3\envs\reddit_env\Lib\site-packages\optuna\_experimental.py:32: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
[I 2026-01-27 18:49:18,810] A new study created in memory with name: no-name-aff8b45d-e8b8-4ce4-be60-dd8e112194cb
[I 2026-01-27 18:49:36,113] Trial 2 finished with value: -1001.958715883967 and parameters: {'window_size': 30, 'n_lags': 1, 'lambda': 0.001698692702620091}. Best is trial 2 with value: -1001.958715883967.
[I 2026-01-27 18:49:45,664] Trial 3 finished with value: -1001.9599994654786 and parameters: {'window_size': 230, 'n_lags': 1, 'lambda': 0.0009885571556753037}. Best is trial 2 with value: -1001.958715883967.
[I 2026-01-27 18:50:01,112] Trial 5 finished with value: -1001.9599991285182 and parameters: {'window_size': 20, 'n_lags': 4, 'lambda': 0.000942898978432539}. Best is trial 2 with value: -1001.958715883967.
[I 2026-01-27 18:50:05,722] Trial 0 finished 

In [6]:
res  = estimate_single_config(
    X, y,
    window_size= 10,
    n_lags=5,
    lambda_val=0.0025995,
    standardize=True,   
    verbose=True,
    return_details=True
)

[I 2026-01-27 21:27:30,732] A new study created in memory with name: no-name-7b4d2e81-67d2-45c2-94ca-3d28caab1ac2
[I 2026-01-27 21:28:27,792] Trial 1 finished with value: -1000.8895117569249 and parameters: {'window_size': 190, 'n_lags': 13, 'lambda': 0.0017511478005244685}. Best is trial 1 with value: -1000.8895117569249.
[I 2026-01-27 21:29:37,981] Trial 6 finished with value: -1001.9599471398107 and parameters: {'window_size': 20, 'n_lags': 15, 'lambda': 0.0008124021842166388}. Best is trial 1 with value: -1000.8895117569249.
[I 2026-01-27 21:29:54,386] Trial 2 finished with value: -1001.9598338424879 and parameters: {'window_size': 70, 'n_lags': 22, 'lambda': 0.00039440253103967704}. Best is trial 1 with value: -1000.8895117569249.
[I 2026-01-27 21:31:32,998] Trial 7 finished with value: -1001.9599502981587 and parameters: {'window_size': 220, 'n_lags': 15, 'lambda': 0.0005351059488234689}. Best is trial 1 with value: -1000.8895117569249.
[I 2026-01-27 21:32:18,538] Trial 5 finishe

In [ ]:
res

In [13]:
trials_df = study.trials_dataframe(
    attrs=("number", "value", "state", "params", "user_attrs", "system_attrs")
)

# save to csv
trials_df.to_csv("optuna_study_trials.csv", index=False)

In [12]:
trials_df

,number,value,state,params_lambda,params_n_lags,params_window_size,user_attrs_kappa,user_attrs_kappa_tstat,user_attrs_r2_stage1,user_attrs_r2_stage2,user_attrs_status
0,0,-1001.959964,COMPLETE,0.000095,17,270,9.492284e-07,3.639570e-05,0.986746,-7.485450e-08,infeasible
1,1,-1000.889512,COMPLETE,0.001751,13,190,7.317783e-02,1.070488e+00,0.128646,5.860510e-04,infeasible
2,2,-1001.959834,COMPLETE,0.000394,22,70,5.452154e-06,1.661575e-04,0.959121,-8.862469e-08,infeasible
3,3,-1001.960000,COMPLETE,0.000036,18,260,8.551859e-09,3.379374e-07,0.996897,-4.980902e-10,infeasible
4,4,-1001.960000,COMPLETE,0.000014,15,260,2.126423e-09,8.717576e-08,0.998224,-1.463361e-10,infeasible
...,...,...,...,...,...,...,...,...,...,...,...
495,495,4.716261,COMPLETE,0.002566,18,300,5.820636e-01,5.404531e+00,0.005289,3.249890e-03,feasible
496,496,-1000.006192,COMPLETE,0.003039,18,300,9.251155e-01,5.068332e+01,-0.000609,9.056133e-03,infeasible
497,497,-1000.000000,COMPLETE,0.007401,17,300,NaN,NaN,NaN,NaN,invalid_nan
498,498,4.801630,COMPLETE,0.002632,17,300,6.376268e-01,6.561543e+00,0.003940,3.597088e-03,feasible


In [ ]:
import numpy as np
import optuna
from optuna.samplers import TPESampler

def objective(trial):
    # Search space
    window_size = trial.suggest_int("window_size", 10, 400, step=10)
    n_lags      = trial.suggest_int("n_lags", 1, 25)
    lam         = trial.suggest_float("lambda", 1e-5, 1e-2, log=True)

    # Run estimation
    res = estimate_single_config(
        X, y, window_size, n_lags, lam,
        standardize=True, verbose=False, return_details=False
    )
    summary = (res or {}).get("summary", {})

    # Primary objective: maximize in-sample R^2 from stage 2
    r2_2 = float(summary.get("r2_insample_stage2", np.nan))

    if not np.isfinite(r2_2):
        raise optuna.TrialPruned()

    trial.set_user_attr("r2_stage2", r2_2)
    trial.set_user_attr("r2_stage1", float(summary.get("r2_insample_stage1", np.nan)))
    trial.set_user_attr("kappa",     float(summary.get("kappa", np.nan)))
    trial.set_user_attr("kappa_tstat", float(summary.get("kappa_tstat", np.nan)))

    return r2_2


sampler = TPESampler(
    seed=42,
    multivariate=True,
    n_startup_trials=50,
    n_ei_candidates=64
)

study1 = optuna.create_study(direction="maximize", sampler=sampler)
study1.optimize(objective, n_trials=300, n_jobs=6, gc_after_trial=True)

c:\Users\jonat\anaconda3\envs\reddit_env\Lib\site-packages\optuna\_experimental.py:32: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
[I 2026-01-28 07:50:34,647] A new study created in memory with name: no-name-91c10e75-27e8-4da6-95c1-4fb494423d0b
[I 2026-01-28 07:51:13,606] Trial 0 finished with value: -2.180994940204073e-09 and parameters: {'window_size': 140, 'n_lags': 4, 'lambda': 0.0015244386293093984}. Best is trial 0 with value: -2.180994940204073e-09.
[I 2026-01-28 07:51:45,887] Trial 2 finished with value: -1.704536511937249e-09 and parameters: {'window_size': 160, 'n_lags': 12, 'lambda': 0.003163567379362874}. Best is trial 2 with value: -1.704536511937249e-09.
[I 2026-01-28 07:51:46,234] Trial 3 finished with value: 0.0006257024969459346 and parameters: {'window_size': 370, 'n_lags': 7, 'lambda': 0.002138413627601193}. Best is trial 3 with value: 0.0006257024969459346.
[I 2026-01-28 07:52:22